# 07. 기준 위성 비교: MODIS 전체 vs Aqua

실제 데이터에서 2024~2025년에 Terra 탐지가 줄면서(Terra ÷ VIIRS 0.077 → 0.056), "MODIS 전체"를 맞히는 모델이
2025년을 6~10% 많게 예측했습니다. Terra는 2020년 이후 궤도 유지를 멈춰 지나가는 시각이 점점 이른 아침으로 밀리고 있습니다.
VIIRS S-NPP와 같은 오후 13:30에 지나가는 **Aqua만** 기준으로 삼으면 두 센서의 관계가 더 안정적인지 비교합니다.

| 기준 | 목표 열 | 뜻 |
| --- | --- | --- |
| MODIS 전체 (현재) | `modis_n` | Terra + Aqua |
| Aqua | `modis_n_aqua` | 오후 13:30 위성만 |
| Terra (참고) | `modis_n_terra` | 오전 위성만. 어느 쪽이 흔들리는지 보려고 같이 돌림 |

모델은 채택한 LightGBM(단조+Tweedie)만 씁니다. 규칙 모델은 MODIS 낮·밤 열을 따로 맞히는데, 위성별 낮·밤 열은 격자에 없습니다.

**미리 정한 판단 기준** (결과를 보기 전에 고정). Aqua는 전체의 약 2/3라 오차를 비율로 비교합니다.
1. 상대 합계 오차(합계 MAE ÷ 평균 실제 합계)가 MODIS 전체 기준보다 **5% 넘게 나빠지지 않을 것**
2. 최근 2년(2024·2025 시험) 평균 |편향|이 MODIS 전체 기준보다 **작을 것**

둘 다 맞으면 Aqua 기준을 추천합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsml import evaluate  # 한글 폰트 설정 포함
from fsml.aggregate import overlap
from fsml.compare import AI, PRIMARY, cross_validate
from fsml.features import drop_partial_periods, make_features
from fsml.paths import default_grid
from fsml.splits import auto_folds, block_id

TARGETS = {"MODIS 전체": "modis_n", "Aqua": "modis_n_aqua", "Terra": "modis_n_terra"}

GRID = default_grid()
grid = pd.read_parquet(GRID)
missing = [c for c in TARGETS.values() if c not in grid]
assert not missing, f"격자에 {missing} 열이 없습니다. Terra/Aqua 열이 있는 aggregate.py로 격자를 다시 만드세요."
res = float(np.round(np.diff(np.sort(grid["lat"].unique())).min(), 6))  # 격자 크기(°)
data, FEATURES = make_features(drop_partial_periods(overlap(grid)).reset_index(drop=True), res)
kind, folds = auto_folds(data, n=5)
assert kind == "time", "이 노트북은 여러 해 실제 격자(grid_0.5_month.parquet)가 있어야 합니다. 7일 샘플로는 돌리지 않습니다."
print(f"{GRID.name} | 검증 방식: {'연도별' if kind == 'time' else '지역 블록별'} {len(folds)}회")

## 연도별 위성 ÷ VIIRS 비율

선이 평평할수록 VIIRS와의 관계가 해마다 일정하다는 뜻이고, 그만큼 환산하기 쉽습니다.

In [ ]:
year = pd.to_datetime(data["period"]).dt.year
yearly = data.groupby(year)[["viirs_n", *TARGETS.values()]].sum()
ratio = pd.DataFrame({name: yearly[col] / yearly["viirs_n"] for name, col in TARGETS.items()})
ax = (ratio / ratio.loc[:2023].mean()).plot(marker="o", figsize=(9, 4))
ax.axhline(1, color="gray", lw=0.8)
ax.set_xlabel("연도")
ax.set_ylabel("÷ VIIRS 비율 (2012~2023 평균 = 1)")
ax.set_title("위성별 VIIRS 대비 탐지 비율");
ratio.round(3)

## 기준별 5회 반복 시험 (LightGBM)

In [ ]:
rows = []
for name, col in TARGETS.items():
    d = data.assign(modis_n=data[col])  # 모델과 지표는 modis_n을 맞히므로 목표 열만 바꿔 넣음
    r = cross_validate(d, FEATURES, folds, models={})
    for k, (_, _, te) in enumerate(folds, start=1):
        t = d[te]
        r.loc[r["fold"] == k, "시험 연도"] = int(pd.to_datetime(t["period"]).dt.year.iloc[0])
        r.loc[r["fold"] == k, "평균 실제 합계"] = t.groupby([block_id(t), "period"])["modis_n"].sum().mean()
    rows.append(r.assign(기준=name))
results = pd.concat(rows, ignore_index=True)
results["시험 연도"] = results["시험 연도"].astype(int)
results["상대 합계 오차"] = results[PRIMARY] / results["평균 실제 합계"]
results.pivot(index="시험 연도", columns="기준", values=["상대 합계 오차", "편향 %"]).round(3)

In [ ]:
ax = results.pivot(index="시험 연도", columns="기준", values="편향 %").plot.bar(figsize=(9, 4), rot=0)
ax.axhline(0, color="gray", lw=0.8)
ax.set_ylabel("편향 % (0에 가까울수록 좋음)")
ax.set_title("시험 연도별 편향: 어떤 기준이 해마다 안정적인가");

## 판정

In [ ]:
recent = results["시험 연도"] >= results["시험 연도"].max() - 1
summary = pd.DataFrame({
    "상대 합계 오차": results.groupby("기준")["상대 합계 오차"].mean(),
    "칸 R²(log)": results.groupby("기준")["칸 R²(log)"].mean(),
    "평균 |편향| %": results.groupby("기준")["편향 %"].apply(lambda s: s.abs().mean()),
    "최근 2년 |편향| %": results[recent].groupby("기준")["편향 %"].apply(lambda s: s.abs().mean()),
}).loc[list(TARGETS)]

base, cand = summary.loc["MODIS 전체"], summary.loc["Aqua"]
checks = {
    "상대 합계 오차가 5% 넘게 나빠지지 않음": cand["상대 합계 오차"] <= base["상대 합계 오차"] * 1.05,
    "최근 2년 |편향|이 더 작음": cand["최근 2년 |편향| %"] < base["최근 2년 |편향| %"],
}
for k, ok in checks.items():
    print(f"{'통과' if ok else '미달'}: {k}")
print("→ Aqua 기준 추천" if all(checks.values()) else "→ MODIS 전체 기준 유지")
summary.round(3)

## 달력에 쓸 때 주의

- **2002-07 이전에는 Aqua가 없습니다.** Aqua 기준이면 2000-11~2002-06(20개월)은 Terra만 있으니, 달력을 2002-07부터 시작하거나
  Terra를 Aqua 기준으로 바꿔 줘야 합니다(Aqua ÷ Terra는 Terra 궤도가 유지되던 2003~2019년에 1.9~2.1로 거의 일정).
- **MODIS 전체 기준도 같은 문제가 있습니다.** 2002-06까지의 "MODIS 전체"는 사실 Terra 하나라서, 2002-07에 탐지 수가 약 3배로 뜁니다.
  이 기간을 평년값에 넣으면 이상 판정 기준이 낮아집니다. 어느 기준을 고르든 2002-07 이전은 따로 처리해야 합니다.
- Aqua도 2022년 무렵부터 궤도 유지를 멈춰 지나가는 시각이 조금씩 늦어지고 있습니다(Terra보다 느림). 해마다 이 노트북을 다시 돌려 비율이 흔들리는지 확인합니다.